# 02 - Control the modelling space

Choice modellers often want to ask focused questions:

- What if I only search transformations?
- What if I only switch between generic and alternative-specific tastes?
- What changes when I add covariates?
- What happens if I change covariate levels?

Delphos lets you do this by creating task copies with a smaller search space.


### 1. Load a task and inspect the default space


In [ ]:
import delphos as dp
from delphos.grammar import build_runtime

agent = dp.load_agent()
task = dp.load_dataset("Swissmetro")

print("Attributes:", task.attribute_names)
print("Transformations:", task.transform_names)
print("Tastes:", task.taste_names)
print("Covariates:", task.covariate_names)


### 2. Helper to count available actions


In [ ]:
def count_task_actions(task, linear_additive=True):
    runtime = build_runtime(
        task=task,
        catalogue=agent.catalogue,
        linear_additive=linear_additive,
        device="cpu",
    )
    return int(runtime.action_space.task_mask.sum().item())

print("Default task actions:", count_task_actions(task))


### 3. Transformation-only search

Keep tastes fixed to generic and remove covariates. Delphos can still choose linear, log, or Box-Cox transformations.


In [ ]:
transformation_task = dp.configure_modelling_space(
    task,
    tastes=["generic"],
    covariates=[],
)

print("Transformations:", transformation_task.transform_names)
print("Tastes:", transformation_task.taste_names)
print("Covariates:", transformation_task.covariate_names)
print("Actions:", count_task_actions(transformation_task))

transformation_models = agent.propose(
    transformation_task,
    n_models=3
)
transformation_models.to_dataframe()


### 4. Taste-only search

Freeze transformations to linear and remove covariates. Delphos then searches generic vs alternative-specific tastes.


In [ ]:
taste_task = dp.configure_modelling_space(
    task,
    transformations=["linear"],
    covariates=[],
)

print("Transformations:", taste_task.transform_names)
print("Tastes:", taste_task.taste_names)
print("Covariates:", taste_task.covariate_names)
print("Actions:", count_task_actions(taste_task))

taste_models = agent.propose(
    taste_task,
    n_models=3,

)
taste_models.to_dataframe()


### 5. Add covariates deliberately

Start with one or two covariates, then expand. This keeps estimation manageable and makes interpretation easier.


In [ ]:
covariate_task = dp.configure_modelling_space(
    task,
    transformations=["linear", "log"],
    tastes=["generic", "specific"],
    covariates=["income", "purpose"],
)

print("Covariates:", covariate_task.covariate_names)
print("Actions:", count_task_actions(covariate_task))

covariate_models = agent.propose(
    covariate_task,
    n_models=3
)
covariate_models.to_dataframe()


### 6. Search only selected attributes

Use an attribute subset when you want a compact model family. `ASC` is attribute id 1 and is commonly kept.


In [ ]:
compact_task = dp.configure_modelling_space(
    task,
    attributes=["ASC", "time", "cost"],
    transformations=["linear", "log"],
    covariates=[],
)

print("Attributes:", compact_task.attribute_names)
print("Actions:", count_task_actions(compact_task))


### 7. Change covariate levels

Covariate levels control how many interaction parameters Apollo creates. Collapsing levels is often useful for quick experiments.


In [ ]:
collapsed_task = dp.set_covariate_levels(
    task,
    income=[1, 2],
    purpose=[1, 2],
)

for cov in collapsed_task.covariates:
    if cov.name in {"income", "purpose"}:
        print(cov.name, cov.levels)


### 8. Recommended workflow

1. Start with transformations only.
2. Add taste variation.
3. Add one covariate family at a time.
4. Estimate a small number of models.
5. Expand the search only after the simple space behaves well.
